#**Golden Solutions of Main Problem and subproblems**

This workflow presents the coding solution to the scientific coding problem and subproblems given to Gemini 3 pro.
Unit tests for the main problem and subproblems are included in separate sections.

**Golden solution of main problem**

In [1]:
import numpy as np
from scipy.optimize import brentq
import math

In [2]:
def get_sequestered_mass(ts, T, d, L, air_flow, pell_flow, v_pell, nCO2, MW):
    """
    Function to calculate the mass of carbon dioxide sequestered over a window of time.
    The calculation uses the Levenspiel Shrinking Core Model (SCM) to determine
    the conversion of calcium hydroxide sorbent pellets in a countercurrent vessel.
    A discrete set of 1000 conversion values is generated by solving the SCM over
    the time domain [1, 259200] seconds, and the exit conversion is found by
    interpolating at the calculated residence time of the pellets.

    Args:
        ts (float): Operating window of sequestration in hours
        T (float): Operating temperature in Celsius
        d (float): Vessel diameter in meters
        L (float): Column height in meters
        air_flow (float): Flow rate of air in kmol/hr
        pell_flow (float): Flow rate of pellet in kmol/hr
        v_pell (float): Volume flow rate of pellet in m^3/s
        nCO2 (float): Mole fraction of CO2 in air
        MW (float): Average molecular weight of the humid air

    Returns:
        sequestered_mass (float): Mass of CO2 captured over the time window t in kilograms
    """
    # Time domain (6 hours in seconds)
    t = np.linspace(0, 6*3600, 500)

    # Parameters (replace with Aspen / literature values)
    rho_s = 2200      # mol/m^3 (solid molar density)
    dp   = 5/1000     # m (pellet diameter)
    #sphericity assumed to be 1 since the pellets are spherical
    sphericity = 1
    e_void = 0.45     # bed void fraction

    k_g   = 0.02      # m/s (gas film coefficient)
    k_r   = 0.01      # m/s (surface reaction constant)
    D_eff = 1e-9      # m^2/s (effective diffusivity)
    ref_mu = 1.827 * 10**(-5)  # reference viscosity of air, unit is Pa.S
    ref_temp = 291.15          # reference temperature, unit is Kelvin
    suther_const = 120         # sutherland constant
    cross_area = math.pi * (d/2)**2                # diameter of DAC column
    kg_per_sec = pell_flow * MW / 3600                  # converts to kg per second
    G_flux = kg_per_sec / cross_area               # Gas mas flux

    #viscosity at the air temperature
    mu  =  ref_mu * ((ref_temp + suther_const) / ((T + 273.15) + suther_const)) * ((T + 273.15) / ref_temp)**1.5

    # Constant values
    R = 8.314            # Universal gas constant, unit is J/(mol.K)

    # Calculate the RT/MW AT TEMP, converts molar weight to kg/mol from kg/kmol or g/mol
    RT_ratio_MW = R * (T + 273.15) / (MW/1000)

    #Calculate mass flux of gas
    G = G_flux



    #laminar term
    laminar_term = (150 * mu * G * (1 - 0.45)**2) / (((dp)**2) * (0.45**3) * (sphericity**2))

    # turbulence term
    turbulence_term = (1.75 * (1 - 0.45) * (G**2)) / ((0.45**3) * (dp) * sphericity)

    #The Ergun Equation
    delta_p_squared = 2 * RT_ratio_MW * L * (laminar_term + turbulence_term)

    #Return pressure into DAC. unit of returned value is atm. Take not in Master python script
    p_in = np.sqrt(delta_p_squared + 101325**2)
    p_out =101325
    P_ave = (2/3) * (p_in**2 + p_out**2 + p_in * p_out) /(p_in + p_out)
    yCO2 = nCO2   # mole fraction of CO2 in air
    C_CO2 = (yCO2 * P_ave) /(R * T) #Molar concentration of CO2 in air %mol/m^3


    # Preallocate arrays
    X = np.zeros_like(t)

    t_surface  = np.zeros_like(t)
    t_reaction = np.zeros_like(t)
    t_diff     = np.zeros_like(t)

    # Loop over time
    for i, ti in enumerate(t):

        # Implicit SCM equation
        def fun(X):
            return (
                (rho_s*(dp/2)/(3*k_g*C_CO2))*X +
                (rho_s*(dp/2)/(k_r*C_CO2))*(1 - (1-X)**(1/3)) +
                (rho_s*((dp/2)**2)/(6*D_eff*C_CO2)) *
                (1 - 3*(1-X)**(2/3) + 2*(1-X))
                - ti
            )

        # Solve conversion
        X[i] = brentq(fun, 0, 1)

    #residence time calculations
    #Parameters definition

    Vreactor = (math.pi * L * (d**2)) /4  #volume of reactor

    #Vvoid calculations
    Vvoid = (1 - e_void) * Vreactor           # Actual gas volume available

    #calculations of the residence time of pellet

    t_res = (Vvoid)/v_pell

    #Interpolating to calculate the carbonation conversion for a certain residence time

    X_res = np.interp(t_res, t, X)
    #print(X_res)

    # Calculating the equivalent conversion with respect to the CO2
    XCO2 = X_res * pell_flow  / (nCO2 * air_flow)
    #print(XCO2)

    #Calculates the mass sequestered over a period of ts
    sequestered_mass = nCO2 * air_flow * ts * XCO2 * 44.01

    return sequestered_mass

#**Unit tests for main problem**

In [3]:
def main_problem_unit_tests():
    # Output existence test
    #Checks if the function call returns a value that is not None
    score_point_main=0
    try:
        assert (get_sequestered_mass(ts=25, T=40, d=9, L=10, air_flow=10000, pell_flow=1000, v_pell=200, nCO2=0.0006, MW=29)) != None
        score_point_main+=1
        print(f"Total of {score_point_main} points scored on output existence tests ✔️")
    except Exception as e:
        print("Failed output existence test")

    #Output correctness test 1
    #Checks if the function outputs the correct value
    try:
        result = get_sequestered_mass(ts=22, T=38, d=7, L=9, air_flow=7000, pell_flow=150, v_pell=100, nCO2=0.00056, MW=19.56)
        assert np.isclose(result, 19.2645, rtol=1e-1)
        score_point_main+=1
    except Exception as e:
        print("Failed output correctness test 1")
        print(f"Expected 19.2645, got {result}")

    #Output correctness test 2
    #Checks if the function outputs the correct value

    try:
        result = get_sequestered_mass(ts=30, T=45, d=5, L=8, air_flow=6000, pell_flow=35, v_pell=2.759e-3, nCO2=0.0004, MW=28.94)
        assert np.isclose(result, 2518.243, rtol=1e-1)
        score_point_main+=1
    except Exception as e:
        print("Failed output correctness test 2")
        print(f"Expected 2518.243, got {result}")


    #Output correctness test 3
    #Checks if the function outputs the correct answer
    try:
        result = get_sequestered_mass(ts=15, T=30, d=6, L=6, air_flow=4000, pell_flow=20, v_pell=4.73e-3, nCO2=0.0004, MW=28.92)
        assert np.isclose(result, 840.6530, rtol=1e-1)
        score_point_main+=1
        print(f"Total of {score_point_main} points scored on output correctness tests and output existence tests✔️")
    except Exception as e:
        print("Failed output correctness test 3")
        print(f"Expected 840.6530, got {result}")


    #Boundary and sanity test
    #Checks to see if the code solution holds up in extreme conditions
    #At time zero, the mass sequestered should be exactly zero
    try:
        result = get_sequestered_mass(ts=0, T=30, d=6, L=6, air_flow=4000, pell_flow=20, v_pell=4.73e-3, nCO2=0.0004, MW=28.92)
        assert np.isclose(result, 0, rtol=1e-1)
        score_point_main+=1
        print(f"Total of {score_point_main} points scored on all tests ✔️")
    except Exception as e:
        print("Failed boundary and sanity test")
        print(f"Expected 0, got {result}")

    #Compute overall percentage score
    percent_score= (score_point_main/5)*100
    print(f"Overall score on main problem is {percent_score} percent")
    return percent_score

In [4]:
main_problem_unit_tests()

Total of 1 points scored on output existence tests ✔️
Total of 4 points scored on output correctness tests and output existence tests✔️
Total of 5 points scored on all tests ✔️
Overall score on main problem is 100.0 percent


100.0

#**Subproblem 1.1 Golden solution**

In [5]:
def get_average_pressure(T, d, L, air_flow, MW):
    """
    Function to calculate the average gas pressure in the reactor.

    Args:
        T (float): The operating temperature in Celsius
        d (float): The vessel diameter in meters
        L (float): The height of the column in meters
        air_flow (float): The flow rate of air in kmol/hr
        MW (float): The average molecular weight of the humid air
        pell_flow (float): The flow rate of pellet in kmol/hr
        dp (float): Particle diameter in mm
        sphericity (float): Particle sphericity

    Returns:
        P_ave (float): average gas pressure in the reactor, in atm
    """
    ref_mu = 1.827 * 10**(-5)  # reference viscosity of air, unit is Pa.S
    ref_temp = 291.15          # reference temperature, unit is Kelvin
    suther_const = 120         # sutherland constant
    cross_area = math.pi * (d / 2)**2
    kg_per_sec = air_flow * MW / 3600  # Using air_flow for gas pressure calculation
    G_flux = kg_per_sec / cross_area
    sphericity = 1
    dp = 5
    # Viscosity at the air temperature
    mu = ref_mu * ((ref_temp + suther_const) / ((T + 273.15) + suther_const)) * ((T + 273.15) / ref_temp)**1.5

    # Constant values
    R = 8.314
    RT_ratio_MW = R * (T + 273.15) / (MW / 1000)
    G = G_flux

    # Laminar and turbulence terms for Ergun Equation
    laminar_term = (150 * mu * G * (1 - 0.45)**2) / (((dp / 1000)**2) * (0.45**3) * (sphericity**2))
    turbulence_term = (1.75 * (1 - 0.45) * (G**2)) / ((0.45**3) * (dp / 1000) * sphericity)

    # Ergun Equation
    delta_p_squared = 2 * RT_ratio_MW * L * (laminar_term + turbulence_term)

    p_in = np.sqrt(delta_p_squared + 101325**2)
    #print(p_in/101325)
    p_out = 101325
    P_ave = (2/3) * (p_in**2 + p_out**2 + p_in * p_out) / (p_in + p_out)

    return P_ave/101325

#**Unit tests for sub problem 1.1**

In [6]:

def subproblem1_unit_tests():
    # Output existence test
    #Checks if the function call returns a value that is not None
    score_point_sub1=0
    try:
        assert (get_average_pressure(T=40, d=9, L=10, air_flow=10000, MW=29)) != None
        score_point_sub1+=1
        print(f"Total of {score_point_sub1} points scored on output existence tests ✔️")
    except Exception as e:
        print("Failed output existence test")

    #Output correctness test 1
    #Checks if the function outputs the correct value
    try:
        result = get_average_pressure(T=38, d=7, L=9, air_flow=7000, MW=19.56)
        assert np.isclose(result, 1.13655, rtol=1e-1)
        score_point_sub1+=1
    except Exception as e:
        print("Failed output correctness test 1")
        print(f"expected 1.13655, got {result}")

    #Output correctness test 2
    #Checks if the function outputs the correct value

    try:
        result = get_average_pressure(T=45, d=5, L=8, air_flow=6000, MW=28.94)
        assert np.isclose(result, 1.39073, rtol=1e-1)
        score_point_sub1+=1
    except Exception as e:
        print("Failed output correctness test 2")
        print(f"expected 1.39073, got {result}")


    #Output correctness test 3
    #Checks if the function outputs the correct answer
    try:
        result = get_average_pressure(T=30, d=6, L=6, air_flow=4000, MW=28.92)
        assert np.isclose(result, 1.07650, rtol=1e-1)
        score_point_sub1+=1
        print(f"Total of {score_point_sub1} points scored on output correctness tests and output existence tests✔️")
    except Exception as e:
        print("Failed output correctness test 3")
        print(f"expected 1.07650, got {result}")


    #Boundary and sanity test
    #Checks to see if the code solution holds up in extreme conditions
    #When there is no air flow, the average gas pressure in the reactor must be atm pressure (1 atm)
    try:
        result = get_average_pressure(T=30, d=6, L=6, air_flow=0, MW=28.92)
        assert np.isclose(result, 1, rtol=1e-1)
        score_point_sub1+=1
        print(f"Total of {score_point_sub1} points scored on all tests ✔️")
    except Exception as e:
        print("Failed boundary and sanity test")
        print(f"expected 1, got {result}")

    #Compute overall percentage score
    percent_score= (score_point_sub1/5)*100
    print(f"Overall score on sub problem 1.1 is {percent_score} percent")
    return percent_score

In [7]:
subproblem1_unit_tests()

Total of 1 points scored on output existence tests ✔️
Total of 4 points scored on output correctness tests and output existence tests✔️
Total of 5 points scored on all tests ✔️
Overall score on sub problem 1.1 is 100.0 percent


100.0

#**Subproblem 1.2 Golden solution**

In [8]:
def get_carbonation_conversion(T, d, L, air_flow, pell_flow, v_pell, nCO2, MW):
    """
    Function to calculate the carbonation conversion of Calcium hydroxide pellets by
    interpolating linear over a set of discrete numerical solution of the Levenspiels
    shrinking core model.

    Args:
        ts (float): Operating window of sequestration in hours
        T (float): Operating temperature in Celsius
        d (float): Vessel diameter in meters
        L (float): Column height in meters
        air_flow (float): Flow rate of air in kmol/hr
        pell_flow (float): Flow rate of pellet in kmol/hr
        v_pell (float): Volume flow rate of pellet in m^3/hr
        nCO2 (float): Mole fraction of CO2 in air
        MW (float): Average molecular weight of the humid air

    Returns:
        sequestered_mass (float): Mass of CO2 captured over the time window t in kilograms
    """
    # Time domain (6 hours in seconds)
    t = np.linspace(0, 6*3600, 500)

    # Parameters (replace with Aspen / literature values)
    rho_s = 2200      # mol/m^3 (solid molar density)
    dp   = 5/1000     # m (pellet diameter)
    #sphericity assumed to be 1 since the pellets are spherical
    sphericity = 1
    e_void = 0.45     # bed void fraction

    k_g   = 0.02      # m/s (gas film coefficient)
    k_r   = 0.01      # m/s (surface reaction constant)
    D_eff = 1e-9      # m^2/s (effective diffusivity)
    ref_mu = 1.827 * 10**(-5)  # reference viscosity of air, unit is Pa.S
    ref_temp = 291.15          # reference temperature, unit is Kelvin
    suther_const = 120         # sutherland constant
    cross_area = math.pi * (d/2)**2                # diameter of DAC column
    kg_per_sec = pell_flow * MW / 3600                  # converts to kg per second
    G_flux = kg_per_sec / cross_area               # Gas mas flux

    #viscosity at the air temperature
    mu  =  ref_mu * ((ref_temp + suther_const) / ((T + 273.15) + suther_const)) * ((T + 273.15) / ref_temp)**1.5

    # Constant values
    R = 8.314            # Universal gas constant, unit is J/(mol.K)

    # Calculate the RT/MW AT TEMP, converts molar weight to kg/mol from kg/kmol or g/mol
    RT_ratio_MW = R * (T + 273.15) / (MW/1000)

    #Calculate mass flux of gas
    G = G_flux



    #laminar term
    laminar_term = (150 * mu * G * (1 - 0.45)**2) / (((dp)**2) * (0.45**3) * (sphericity**2))

    # turbulence term
    turbulence_term = (1.75 * (1 - 0.45) * (G**2)) / ((0.45**3) * (dp) * sphericity)

    #The Ergun Equation
    delta_p_squared = 2 * RT_ratio_MW * L * (laminar_term + turbulence_term)

    #Return pressure into DAC. unit of returned value is atm. Take not in Master python script
    p_in = np.sqrt(delta_p_squared + 101325**2)
    p_out =101325
    P_ave = (2/3) * (p_in**2 + p_out**2 + p_in * p_out) /(p_in + p_out)
    yCO2 = nCO2   # mole fraction of CO2 in air
    C_CO2 = (yCO2 * P_ave) /(R * T) #Molar concentration of CO2 in air %mol/m^3


    # Preallocate arrays
    X = np.zeros_like(t)

    t_surface  = np.zeros_like(t)
    t_reaction = np.zeros_like(t)
    t_diff     = np.zeros_like(t)

    # Loop over time
    for i, ti in enumerate(t):

        # Implicit SCM equation
        def fun(X):
            return (
                (rho_s*(dp/2)/(3*k_g*C_CO2))*X +
                (rho_s*(dp/2)/(k_r*C_CO2))*(1 - (1-X)**(1/3)) +
                (rho_s*((dp/2)**2)/(6*D_eff*C_CO2)) *
                (1 - 3*(1-X)**(2/3) + 2*(1-X))
                - ti
            )

        # Solve conversion
        X[i] = brentq(fun, 0, 1)

    #residence time calculations
    #Parameters definition

    Vreactor = (math.pi * L * (d**2)) /4  #volume of reactor

    #Vvoid calculations
    Vvoid = (1 - e_void) * Vreactor           # Actual gas volume available

    #calculations of the residence time of pellet

    t_res = (Vvoid)/v_pell

    #Interpolating to calculate the carbonation conversion for a certain residence time

    X_res = np.interp(t_res, t, X)
    return X_res

#**Unit tests for sub problem 1.2**

In [9]:

def subproblem2_unit_tests():
    # Output existence test
    #Checks if the function call returns a value that is not None
    score_point_sub2=0
    try:
        assert (get_carbonation_conversion(T=40, d=9, L=10, air_flow=10000, pell_flow=1000, v_pell=200, nCO2=0.0006, MW=29)) != None
        score_point_sub2+=1
        print(f"Total of {score_point_sub2} points scored on output existence tests ✔️")
    except Exception as e:
        print("Failed output existence test")

    #Output correctness test 1
    #Checks if the function outputs the correct value
    try:
        result = get_carbonation_conversion(T=38, d=7, L=9, air_flow=7000, pell_flow=150, v_pell=100, nCO2=0.00056, MW=19.56)
        assert np.isclose(result, 0.000133, rtol=1e-1)
        score_point_sub2+=1
    except Exception as e:
        print("Failed output correctness test 1")
        print(f"Expected 0.000133, got {result}")

    #Output correctness test 2
    #Checks if the function outputs the correct value

    try:
        result = get_carbonation_conversion(T=45, d=5, L=8, air_flow=6000, pell_flow=35, v_pell=2.759e-3, nCO2=0.0004, MW=28.94)
        assert np.isclose(result, 0.05449, rtol=1e-1)
        score_point_sub2+=1
    except Exception as e:
        print("Failed output correctness test 2")
        print(f"Expected 0.05449, got {result}")


    #Output correctness test 3
    #Checks if the function outputs the correct answer
    try:
        result = get_carbonation_conversion(T=30, d=6, L=6, air_flow=4000, pell_flow=20, v_pell=4.73e-3, nCO2=0.0004, MW=28.92)
        assert np.isclose(result, 0.06367, rtol=1e-1)
        score_point_sub2+=1
        print(f"Total of {score_point_sub2} points scored on output correctness tests and output existence tests✔️")
    except Exception as e:
        print("Failed output correctness test 3")
        print(f"Expected 0.06367, got {result}")


    #Boundary and sanity test
    #Checks to see if the code solution holds up in extreme conditions
    #As the volume of pellet becomes absurdly high, the carbonation conversion should be approximately zero
    try:
        result = get_carbonation_conversion(T=30, d=6, L=6, air_flow=6000, pell_flow=30, v_pell=4.73e30, nCO2=0.0004, MW=28.92)
        assert np.isclose(round(result, 3), 0, rtol=1e-3)
        score_point_sub2+=1
        print(f"Total of {score_point_sub2} points scored on all tests ✔️")
    except Exception as e:
        print("Failed boundary and sanity test")
        print(f"Expected 0, got {result}")

    #Compute overall percentage score
    percent_score= (score_point_sub2/5)*100
    print(f"Overall score on subproblem 1.2 is {percent_score} percent")
    return percent_score

In [10]:
subproblem2_unit_tests()

Total of 1 points scored on output existence tests ✔️
Total of 4 points scored on output correctness tests and output existence tests✔️
Total of 5 points scored on all tests ✔️
Overall score on subproblem 1.2 is 100.0 percent


100.0

#**Golden solution of subproblem 1.3**

In [12]:
def get_sequestered_mass(ts, T, d, L, air_flow, pell_flow, v_pell, nCO2, MW):
    """
    Function to calculate the mass of carbon dioxide sequestered over a window of time.
    The calculation uses the Levenspiel Shrinking Core Model (SCM) to determine
    the conversion of calcium hydroxide sorbent pellets in a countercurrent vessel.
    A discrete set of 1000 conversion values is generated by solving the SCM over
    the time domain [1, 259200] seconds, and the exit conversion is found by
    interpolating at the calculated residence time of the pellets.

    Args:
        ts (float): Operating window of sequestration in hours
        T (float): Operating temperature in Celsius
        d (float): Vessel diameter in meters
        L (float): Column height in meters
        air_flow (float): Flow rate of air in kmol/hr
        pell_flow (float): Flow rate of pellet in kmol/hr
        v_pell (float): Volume flow rate of pellet in m^3/s
        nCO2 (float): Mole fraction of CO2 in air
        MW (float): Average molecular weight of the humid air

    Returns:
        sequestered_mass (float): Mass of CO2 captured over the time window t in kilograms
    """
    # Time domain (6 hours in seconds)
    t = np.linspace(0, 6*3600, 500)

    # Parameters (replace with Aspen / literature values)
    rho_s = 2200      # mol/m^3 (solid molar density)
    dp   = 5/1000     # m (pellet diameter)
    #sphericity assumed to be 1 since the pellets are spherical
    sphericity = 1
    e_void = 0.45     # bed void fraction

    k_g   = 0.02      # m/s (gas film coefficient)
    k_r   = 0.01      # m/s (surface reaction constant)
    D_eff = 1e-9      # m^2/s (effective diffusivity)
    ref_mu = 1.827 * 10**(-5)  # reference viscosity of air, unit is Pa.S
    ref_temp = 291.15          # reference temperature, unit is Kelvin
    suther_const = 120         # sutherland constant
    cross_area = math.pi * (d/2)**2                # diameter of DAC column
    kg_per_sec = pell_flow * MW / 3600                  # converts to kg per second
    G_flux = kg_per_sec / cross_area               # Gas mas flux

    #viscosity at the air temperature
    mu  =  ref_mu * ((ref_temp + suther_const) / ((T + 273.15) + suther_const)) * ((T + 273.15) / ref_temp)**1.5

    # Constant values
    R = 8.314            # Universal gas constant, unit is J/(mol.K)

    # Calculate the RT/MW AT TEMP, converts molar weight to kg/mol from kg/kmol or g/mol
    RT_ratio_MW = R * (T + 273.15) / (MW/1000)

    #Calculate mass flux of gas
    G = G_flux



    #laminar term
    laminar_term = (150 * mu * G * (1 - 0.45)**2) / (((dp)**2) * (0.45**3) * (sphericity**2))

    # turbulence term
    turbulence_term = (1.75 * (1 - 0.45) * (G**2)) / ((0.45**3) * (dp) * sphericity)

    #The Ergun Equation
    delta_p_squared = 2 * RT_ratio_MW * L * (laminar_term + turbulence_term)

    #Return pressure into DAC. unit of returned value is atm. Take not in Master python script
    p_in = np.sqrt(delta_p_squared + 101325**2)
    p_out =101325
    P_ave = (2/3) * (p_in**2 + p_out**2 + p_in * p_out) /(p_in + p_out)
    yCO2 = nCO2   # mole fraction of CO2 in air
    C_CO2 = (yCO2 * P_ave) /(R * T) #Molar concentration of CO2 in air %mol/m^3


    # Preallocate arrays
    X = np.zeros_like(t)

    t_surface  = np.zeros_like(t)
    t_reaction = np.zeros_like(t)
    t_diff     = np.zeros_like(t)

    # Loop over time
    for i, ti in enumerate(t):

        # Implicit SCM equation
        def fun(X):
            return (
                (rho_s*(dp/2)/(3*k_g*C_CO2))*X +
                (rho_s*(dp/2)/(k_r*C_CO2))*(1 - (1-X)**(1/3)) +
                (rho_s*((dp/2)**2)/(6*D_eff*C_CO2)) *
                (1 - 3*(1-X)**(2/3) + 2*(1-X))
                - ti
            )

        # Solve conversion
        X[i] = brentq(fun, 0, 1)

    #residence time calculations
    #Parameters definition

    Vreactor = (math.pi * L * (d**2)) /4  #volume of reactor

    #Vvoid calculations
    Vvoid = (1 - e_void) * Vreactor           # Actual gas volume available

    #calculations of the residence time of pellet

    t_res = (Vvoid)/v_pell

    #Interpolating to calculate the carbonation conversion for a certain residence time

    X_res = np.interp(t_res, t, X)
    #print(X_res)

    # Calculating the equivalent conversion with respect to the CO2
    XCO2 = X_res * pell_flow  / (nCO2 * air_flow)
    #print(XCO2)

    #Calculates the mass sequestered over a period of ts
    sequestered_mass = nCO2 * air_flow * ts * XCO2 * 44.01

    return sequestered_mass

#**Unit tests for sub problem 1.3**

In [13]:

def subproblem3_unit_test():
    # Output existence test
    #Checks if the function call returns a value that is not None
    score_point_sub3=0
    try:
        assert (get_sequestered_mass(ts=25, T=40, d=9, L=10, air_flow=10000, pell_flow=1000, v_pell=200, nCO2=0.0006, MW=29)) != None
        score_point_sub3+=1
        print(f"Total of {score_point_sub3} points scored on output existence tests ✔️")
    except Exception as e:
        print("Failed output existence test")

    #Output correctness test 1
    #Checks if the function outputs the correct value
    try:
        result = get_sequestered_mass(ts=22, T=38, d=7, L=9, air_flow=7000, pell_flow=150, v_pell=100, nCO2=0.00056, MW=19.56)
        assert np.isclose(result, 19.2645, rtol=1e-1)
        score_point_sub3+=1
    except Exception as e:
        print("Failed output correctness test 1")
        print(f"Expected 19.2645, got {result}")

    #Output correctness test 2
    #Checks if the function outputs the correct value

    try:
        result = get_sequestered_mass(ts=30, T=45, d=5, L=8, air_flow=6000, pell_flow=35, v_pell=2.759e-3, nCO2=0.0004, MW=28.94)
        assert np.isclose(result, 2518.243, rtol=1e-1)
        score_point_sub3+=1
    except Exception as e:
        print("Failed output correctness test 2")
        print(f"Expected 2518.243, got {result}")


    #Output correctness test 3
    #Checks if the function outputs the correct answer
    try:
        result = get_sequestered_mass(ts=15, T=30, d=6, L=6, air_flow=4000, pell_flow=20, v_pell=4.73e-3, nCO2=0.0004, MW=28.92)
        assert np.isclose(result, 840.6530, rtol=1e-1)
        score_point_sub3+=1
        print(f"Total of {score_point_sub3} points scored on output correctness tests and output existence tests✔️")
    except Exception as e:
        print("Failed output correctness test 3")
        print(f"Expected 840.6530, got {result}")


    #Boundary and sanity test
    #Checks to see if the code solution holds up in extreme conditions
    #At time zero, the mass sequestered should be exactly zero
    try:
        result = get_sequestered_mass(ts=0, T=30, d=6, L=6, air_flow=4000, pell_flow=20, v_pell=4.73e-3, nCO2=0.0004, MW=28.92)
        assert np.isclose(result, 0, rtol=1e-1)
        score_point_sub3+=1
        print(f"Total of {score_point_sub3} points scored on all tests ✔️")
    except Exception as e:
        print("Failed boundary and sanity test")
        print(f"Expected 0, got {result}")

    #Compute overall percentage score
    percent_score= (score_point_sub3/5)*100
    print(f"Overall score on sub problem 1.3 is {percent_score} percent")
    return percent_score

In [14]:
subproblem3_unit_test()

Total of 1 points scored on output existence tests ✔️
Total of 4 points scored on output correctness tests and output existence tests✔️
Total of 5 points scored on all tests ✔️
Overall score on sub problem 1.3 is 100.0 percent


100.0